In [11]:
#Exercitiul 1 a)
import numpy as np
from hmmlearn import hmm

states = ["1", "2","3","4"]
start_probs = np.array([0.25, 0.25, 0.25,0.25])

# For each topic, the probability of saying certain words can be modeled by
# a distribution over vocabulary associated with the categories

vocabulary = ["Red", "Green", "Blue"]


emission_probs = np.array([[0.7, 0.2, 0.1],
                           [0.2, 0.6, 0.2],
                           [0.1, 0.3, 0.6],
                           [0.5, 0.2, 0.3]
                           ])

# Also assume it's more likely to stay in a state than transition to the other
trans_mat = np.array([[0.2, 0.4, 0, 0.4],
                      [0.4, 0.2, 0.4, 0],
                      [0, 0.4, 0.2, 0.4],
                      [0.4, 0, 0.4, 0.2]])
observable=["Red","Green","Blue","Red"]



vocab2id = {word: i for i, word in enumerate(vocabulary)}
id2topic = dict(zip(range(len(states)), states))
obs_indices = [vocab2id[obs] for obs in observable]

def viterbi(obs_idx, states, start_p, trans_p, emit_p):
    num_states = len(states)
    num_obs = len(obs_idx)

    # Viterbi matrix: viterbi_table[s, t] will store the probability of the most likely
    # path ending in state s at time t
    viterbi_table = np.zeros((num_states, num_obs))

    # Backpointer matrix to keep track of the best path
    backpointers = np.zeros((num_states, num_obs), dtype=int)

    # 1. Initialization (t=0)
    # P(State) * P(Observation | State)
    first_obs = obs_idx[0]
    viterbi_table[:, 0] = start_p * emit_p[:, first_obs]

    # 2. Recursion (t=1 to num_obs-1)
    for t in range(1, num_obs):
        current_obs = obs_idx[t]
        for s in range(num_states):
            # Calculate the probability of reaching state s at time t from all previous states
            # transition_probs: viterbi_table[:, t-1] * trans_mat[:, s]
            # emission_prob: emit_p[s, current_obs]

            prob_paths = viterbi_table[:, t-1] * trans_p[:, s] * emit_p[s, current_obs]

            # Find the maximum probability and its corresponding previous state
            best_prev_state = np.argmax(prob_paths)

            viterbi_table[s, t] = prob_paths[best_prev_state]
            backpointers[s, t] = best_prev_state

    # 3. Termination
    # Find the best final state
    best_last_state = np.argmax(viterbi_table[:, -1])
    best_path_prob = viterbi_table[best_last_state, -1]

    # 4. Backtracking
    # Follow the backpointers to reconstruct the best path
    best_path = [best_last_state]
    for t in range(num_obs - 1, 0, -1):
        best_prev_state = backpointers[best_path[0], t]
        best_path.insert(0, best_prev_state)

    return best_path, best_path_prob

# Run the algorithm
best_path_indices, best_prob = viterbi(obs_indices, states, start_probs, trans_mat, emission_probs)

# Convert state indices back to state names
most_likely_sequence = [id2topic[idx] for idx in best_path_indices]

print("Observation sequence:", observable)
print("Most likely hidden states:", most_likely_sequence)
print(f"Probability of this sequence: {best_prob}")



Observation sequence: ['Red', 'Green', 'Blue', 'Red']
Most likely hidden states: ['1', '2', '3', '4']
Probability of this sequence: 0.002016


In [13]:
#Exercitiul 1 b)
trans_matrix = np.array([[0, 0.5, 0, 0.5],
                      [0.5, 0, 0.5, 0],
                      [0, 0.5, 0, 0.5],
                      [0.5, 0, 0.5, 0]])
best_path_index_modified, best_prob_modified = viterbi(obs_indices,states,start_probs,trans_matrix,emission_probs)
most_likely_sequence_modified = [id2topic[idx] for idx in best_path_index_modified]

print("Observation sequence:", observable)
print("Most likely hidden states:", most_likely_sequence_modified)
print(f"Probability of this sequence: {best_prob_modified}")

print("Dupa modificari avem o probabilitate mai mare a celei mai probabile stari de:",best_prob_modified-best_prob)

Observation sequence: ['Red', 'Green', 'Blue', 'Red']
Most likely hidden states: ['1', '2', '3', '4']
Probability of this sequence: 0.0039375
Dupa modificari avem o probabilitate mai mare a celei mai probabile stari de: 0.0019215


In [14]:
#Exercitiul 2
import pyagrum as pa
import pyagrum.lib.notebook as gnb
import networkx as nx
from IPython.display import display

#a)


# --- 1. PARTEA VIZUALĂ (pyAgrum) ---
uga = pa.UndiGraph()

A = uga.addNode()
B = uga.addNode()
C = uga.addNode()
D = uga.addNode()
E = uga.addNode()

uga.addEdge(A, B)
uga.addEdge(B, C)
uga.addEdge(C, D)
uga.addEdge(D, E)


print("Structura grafului rețelei Markov:")
display(uga)

# --- 2. PARTEA MATEMATICĂ (NetworkX pentru clici reale) ---
Ga = nx.Graph()

# Adăugăm direct muchiile specificate în enunț
Ga.add_edges_from([
    ('A', 'B'), 
    ('B', 'C'),
    ('C', 'D'), 
    ('D', 'E')
])

# Funcția find_cliques din networkx găsește clicile maximale fără să modifice/trianguleze graful
clici_maximale_reale = list(nx.find_cliques(Ga))

for clica in clici_maximale_reale:
    print(f" - {set(clica)}")


Structura grafului rețelei Markov:


(pyagrum.UndiGraph@0x149b9cc0) {0,1,2,3,4} , {0--1,2--3,1--2,3--4}

 - {'E', 'D'}
 - {'B', 'A'}
 - {'B', 'C'}
 - {'D', 'C'}


In [ ]:
#b)
# --- 1. PARTEA VIZUALĂ (pyAgrum) ---
ugb = pa.UndiGraph()

A = ugb.addNode()
B = ugb.addNode()
C = ugb.addNode()
D = ugb.addNode()
E = ugb.addNode()

ugb.addEdge(A, B)
ugb.addEdge(B, C)
ugb.addEdge(C, A)
ugb.addEdge(B, D)
ugb.addEdge(C,E)


print("Structura grafului rețelei Markov:")
display(ugb)

# --- 2. PARTEA MATEMATICĂ (NetworkX pentru clici reale) ---
Gb = nx.Graph()

# Adăugăm direct muchiile specificate în enunț
Gb.add_edges_from([
    ('A', 'B'), 
    ('B', 'C'),
    ('C', 'A'), 
    ('B', 'D'),
    ('C','E')
])

# Funcția find_cliques din networkx găsește clicile maximale fără să modifice/trianguleze graful
clici_maximale_reale = list(nx.find_cliques(Gb))

for clica in clici_maximale_reale:
    print(f" - {set(clica)}")


Structura grafului rețelei Markov:


(pyagrum.UndiGraph@0x142243c0) {0,1,2,3,4} , {0--1,1--3,1--2,0--2,2--4}

 - {'E', 'C'}
 - {'B', 'D'}
 - {'B', 'A', 'C'}


In [ ]:
#Exercitiul 3
import pymc as pm
import arviz as az

with pm.Model() as Model_1:
    theta = pm.Beta("theta", alpha=2, beta=8)
    k = pm.Binomial("k", n=20, p=theta, observed=5)
    idata = pm.sample(data=1000,tune=1000)
print(az.summary(idata,var_names=["theta"]))

az.plot_posterior(idata)

# with pm.Model() as Model_2:
#     theta = pm.Beta("theta", alpha=2, beta=8)
#     k = pm.Binomial("k", n=15, p=theta, observed=3)
#     idata = pm.sample(data=1000,tune=1000)
# print(az.summary(idata,var_names=["theta"]))
# az.plot_posterior(idata)




Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [theta]


Output()

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 2 seconds.


TypeError: cannot directly convert a DataTree into a numpy array. Instead, create an xarray.DataArray first, either with indexing on the DataTree or by invoking the `to_array()` method.